In [1]:
# Copyright (c) 2022-2025, The Isaac Lab Project Developers.
# All rights reserved.
#
# SPDX-License-Identifier: BSD-3-Clause

"""Script to train RL agent with RSL-RL."""

"""Launch Isaac Sim Simulator first."""

import argparse
import sys

from isaaclab.app import AppLauncher

# local imports
import cli_args  # isort: skip

[Warning] [simulation_app] Interactive python shell detected but ISAAC_JUPYTER_KERNEL was not set. Problems with asyncio may occur
[Warning] [simulation_app] Please use Isaac Sim Python 3 kernel instead of the default Python 3 Kernel


In [2]:
from typing import Any

_APPLAUNCHER_CFG_INFO: dict[str, tuple[list[type], Any]] = {
"headless": ([bool], True),
"livestream": ([int], -1),
"enable_cameras": ([bool], False),
"device": ([str], "cuda:0"),
"experience": ([str], ""),
}

In [3]:
# add argparse arguments
parser = argparse.ArgumentParser(description="Train an RL agent with RSL-RL.")
parser.add_argument("--video", action="store_true", default=False, help="Record videos during training.")
parser.add_argument("--video_length", type=int, default=200, help="Length of the recorded video (in steps).")
parser.add_argument("--video_interval", type=int, default=2000, help="Interval between video recordings (in steps).")
parser.add_argument("--num_envs", type=int, default=20, help="Number of environments to simulate.")
parser.add_argument("--task", type=str, default="Isaac-Velocity-Rough-H1-v0", help="Name of the task.")
parser.add_argument("--seed", type=int, default=42, help="Seed used for the environment")
parser.add_argument("--max_iterations", type=int, default=10000, help="RL Policy training iterations.")
arg_group = parser.add_argument_group(
    "app_launcher arguments",
    description="Arguments for the AppLauncher. For more details, please check the documentation.",
)
arg_group.add_argument(
    "--headless",
    action="store_true",
    default=AppLauncher._APPLAUNCHER_CFG_INFO["headless"][1],
    help="Force display off at all times.",
)
arg_group.add_argument(
    "--livestream",
    type=int,
    default=AppLauncher._APPLAUNCHER_CFG_INFO["livestream"][1],
    choices={0, 1, 2},
    help="Force enable livestreaming. Mapping corresponds to that for the `LIVESTREAM` environment variable.",
)
arg_group.add_argument(
    "--enable_cameras",
    action="store_true",
    default=AppLauncher._APPLAUNCHER_CFG_INFO["enable_cameras"][1],
    help="Enable camera sensors and relevant extension dependencies.",
)
arg_group.add_argument(
    "--device",
    type=str,
    default=AppLauncher._APPLAUNCHER_CFG_INFO["device"][1],
    help='The device to run the simulation on. Can be "cpu", "cuda", "cuda:N", where N is the device ID',
)
# Add the deprecated cpu flag to raise an error if it is used
arg_group.add_argument("--cpu", action="store_true", help=argparse.SUPPRESS)
arg_group.add_argument(
    "--verbose",  # Note: This is read by SimulationApp through sys.argv
    action="store_true",
    help="Enable verbose-level log output from the SimulationApp.",
)
arg_group.add_argument(
    "--info",  # Note: This is read by SimulationApp through sys.argv
    action="store_true",
    help="Enable info-level log output from the SimulationApp.",
)
arg_group.add_argument(
    "--experience",
    type=str,
    default="",
    help=(
        "The experience file to load when launching the SimulationApp. If an empty string is provided,"
        " the experience file is determined based on the headless flag. If a relative path is provided,"
        " it is resolved relative to the `apps` folder in Isaac Sim and Isaac Lab (in that order)."
    ),
)
arg_group.add_argument(
    "--kit_args",
    type=str,
    default="",
    help=(
        "Command line arguments for Omniverse Kit as a string separated by a space delimiter."
        ' Example usage: --kit_args "--ext-folder=/path/to/ext1 --ext-folder=/path/to/ext2"'
    ),
)



_StoreAction(option_strings=['--kit_args'], dest='kit_args', nargs=None, const=None, default='', type=<class 'str'>, choices=None, required=False, help='Command line arguments for Omniverse Kit as a string separated by a space delimiter. Example usage: --kit_args "--ext-folder=/path/to/ext1 --ext-folder=/path/to/ext2"', metavar=None)

In [4]:
# append RSL-RL cli arguments
cli_args.add_rsl_rl_args(parser)

# append AppLauncher cli args
args_cli, hydra_args = parser.parse_known_args()

In [5]:

# always enable cameras to record video
if args_cli.video:
    args_cli.enable_cameras = True

# clear out sys.argv for Hydra
sys.argv = [sys.argv[0]] + hydra_args

In [6]:
args_cli.headless = True

In [7]:
import nest_asyncio
nest_asyncio.apply()

# launch omniverse app
app_launcher = AppLauncher(args_cli)
simulation_app = app_launcher.app


[INFO][AppLauncher]: Loading experience file: /home/ksachdev/IsaacLab/apps/isaaclab.python.headless.kit
[Warning] [simulation_app.simulation_app] Modules: ['omni.kit_app'] were loaded before SimulationApp was started and might not be loaded correctly.
[Warning] [simulation_app.simulation_app] Please check to make sure no extra omniverse or pxr modules are imported before the call to SimulationApp(...)
Loading user config located at: '/home/ksachdev/env_isaaclab/lib/python3.10/site-packages/omni/data/Kit/Isaac-Sim/4.5/user.config.json'
[Info] [carb] Logging to file: /home/ksachdev/env_isaaclab/lib/python3.10/site-packages/omni/logs/Kit/Isaac-Sim/4.5/kit_20250206_163817.log
2025-02-06 15:38:17 s] [Warning] [omni.kit.app.plugin] No crash reporter present, dumps uploading isn't available.
2025-02-06 15:38:17 s] [Warning] [omni.usd_config.extension] Enable omni.materialx.libs extension to use MaterialX
2025-02-06 15:38:17 s] [Warning] [omni.isaac.dynamic_control] omni.isaac.dynamic_control 


|---------------------------------------------------------------------------------------------|
| Driver Version: 560.35.03     | Graphics API: Vulkan
|=============================================================================================|
| GPU | Name                             | Active | LDA | GPU Memory | Vendor-ID | LUID       |
|     |                                  |        |     |            | Device-ID | UUID       |
|     |                                  |        |     |            | Bus-ID    |            |
|---------------------------------------------------------------------------------------------|
| 0   | NVIDIA RTX 3500 Ada Generation.. | Yes: 0 |     | 12282   MB | 10de      | 0          |
|     |                                  |        |     |            | 27bb      | eda96a8d.. |
|     |                                  |        |     |            | 1         |            |
|-------------------------------------------------------------------------------

In [8]:
args_cli.__dict__

{'video': False,
 'video_length': 200,
 'video_interval': 2000,
 'num_envs': 20,
 'task': 'Isaac-Velocity-Rough-H1-v0',
 'seed': 42,
 'max_iterations': 10000,
 'device': 'cuda:0',
 'cpu': False,
 'verbose': False,
 'info': False,
 'kit_args': '',
 'experiment_name': None,
 'run_name': None,
 'resume': None,
 'load_run': None,
 'checkpoint': None,
 'logger': None,
 'log_project_name': None,
 'headless': True,
 'hide_ui': True,
 'create_new_stage': False,
 'physics_gpu': 0,
 'active_gpu': 0}

In [9]:
"""Rest everything follows."""

import gymnasium as gym
import os
import torch
from datetime import datetime

from isaaclab_rl.rsl_rl import RslRlOnPolicyRunnerCfg, RslRlVecEnvWrapper
from rsl_rl.runners import OnPolicyRunner

from isaaclab.envs import (
    DirectMARLEnv,
    DirectMARLEnvCfg,
    DirectRLEnvCfg,
    ManagerBasedRLEnvCfg,
    multi_agent_to_single_agent,
)
from isaaclab.utils.dict import print_dict
from isaaclab.utils.io import dump_pickle, dump_yaml

import isaaclab_tasks  # noqa: F401
from isaaclab_tasks.utils import get_checkpoint_path
from isaaclab_tasks.utils.hydra import register_task_to_hydra

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = False

def main(env_cfg: ManagerBasedRLEnvCfg | DirectRLEnvCfg | DirectMARLEnvCfg, agent_cfg: RslRlOnPolicyRunnerCfg):
    """Train with RSL-RL agent."""
    # override configurations with non-hydra CLI arguments
    agent_cfg = cli_args.update_rsl_rl_cfg(agent_cfg, args_cli)
    env_cfg.scene.num_envs = args_cli.num_envs if args_cli.num_envs is not None else env_cfg.scene.num_envs
    agent_cfg.max_iterations = (
        args_cli.max_iterations if args_cli.max_iterations is not None else agent_cfg.max_iterations
    )

    # set the environment seed
    # note: certain randomizations occur in the environment initialization so we set the seed here
    env_cfg.seed = agent_cfg.seed
    env_cfg.sim.device = args_cli.device if args_cli.device is not None else env_cfg.sim.device

    # specify directory for logging experiments
    log_root_path = os.path.join("logs", "rsl_rl", agent_cfg.experiment_name)
    log_root_path = os.path.abspath(log_root_path)
    print(f"[INFO] Logging experiment in directory: {log_root_path}")
    # specify directory for logging runs: {time-stamp}_{run_name}
    log_dir = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    # This way, the Ray Tune workflow can extract experiment name.
    print(f"Exact experiment name requested from command line: {log_dir}")
    if agent_cfg.run_name:
        log_dir += f"_{agent_cfg.run_name}"
    log_dir = os.path.join(log_root_path, log_dir)

    # create isaac environment
    env = gym.make(args_cli.task, cfg=env_cfg, render_mode="rgb_array" if args_cli.video else None)

    # convert to single-agent instance if required by the RL algorithm
    if isinstance(env.unwrapped, DirectMARLEnv):
        env = multi_agent_to_single_agent(env)

    # save resume path before creating a new log_dir
    if agent_cfg.resume:
        resume_path = get_checkpoint_path(log_root_path, agent_cfg.load_run, agent_cfg.load_checkpoint)

    # wrap for video recording
    if args_cli.video:
        video_kwargs = {
            "video_folder": os.path.join(log_dir, "videos", "train"),
            "step_trigger": lambda step: step % args_cli.video_interval == 0,
            "video_length": args_cli.video_length,
            "disable_logger": True,
        }
        print("[INFO] Recording videos during training.")
        print_dict(video_kwargs, nesting=4)
        env = gym.wrappers.RecordVideo(env, **video_kwargs)

    # wrap around environment for rsl-rl
    env = RslRlVecEnvWrapper(env)

    # create runner from rsl-rl
    runner = OnPolicyRunner(env, agent_cfg.to_dict(), log_dir=log_dir, device=agent_cfg.device)
    # write git state to logs
    runner.add_git_repo_to_log(__file__)
    # load the checkpoint
    if agent_cfg.resume:
        print(f"[INFO]: Loading model checkpoint from: {resume_path}")
        # load previously trained model
        runner.load(resume_path)

    # dump the configuration into log-directory
    dump_yaml(os.path.join(log_dir, "params", "env.yaml"), env_cfg)
    dump_yaml(os.path.join(log_dir, "params", "agent.yaml"), agent_cfg)
    dump_pickle(os.path.join(log_dir, "params", "env.pkl"), env_cfg)
    dump_pickle(os.path.join(log_dir, "params", "agent.pkl"), agent_cfg)

    # run training
    runner.learn(num_learning_iterations=agent_cfg.max_iterations, init_at_random_ep_len=True)

    # close the simulator
    env.close()

In [10]:
__file__ = os.getcwd()

In [ ]:
env_cfg, agent_cfg = register_task_to_hydra(args_cli.task, "rsl_rl_cfg_entry_point")
main(env_cfg, agent_cfg)

# close sim app
simulation_app.close()


KeyboardInterrupt: 